In [ ]:
snapshot = SimpleLogitsSnapshot(logits, x_denoising, y_denoising, id_mask)
conf_snapshot = snapshot.transform_logits(collector)
idx_sorted_by_conf = sorter.argsort(conf_snapshot, snapshot)
num_unmask = quota_helper.get_quota(step)
idx_transform = idx_sorted_by_conf[:, :num_unmask]

snapshot.materialize_by_idx_(idx_transform, conf_snapshot)
snapshot.update_this(1, idx_transform, y=x).update_this(1, idx_transform, p_finalized=p_finalized)

In [ ]:

import torch

len_prompt = 0
num_blocks = 0
x = y = plugin_cache_attn = None


# 正常的full一次
# 正常的拿到那个idx_transform
idx_refresh = torch.tensor([], dtype=torch.long, device=x.device)
p_finalized = torch.zeros(x.shape, dtype=torch.float64, device=x.device)

position_start, position_end = 0, len_prompt
idx_block_abs = torch.arange(position_start, position_end, dtype=torch.long, device=x.device)
shape_target = (x.shape[0], position_end, -1)


### 还要处理snapshot扩展的问题，下午处理一下，晚上debug???

for step in range(step_per_block):

    if step == 0 or step_refresh_remainder == 0:
        idx_denoising_abs = idx_block_abs
        idx_current = torch.cat([idx_refresh, idx_denoising_abs])

        logits = model(x[:, idx_current], idx_current=idx_current, shape_target=shape_target).logits
        snapshot = SimpleLogitsSnapshot(logits, x[:, idx_current], y[:, idx_current], id_mask)
        conf_snapshot = snapshot.transform_logits(collector)    # 全的 ### 这里有问题

    else:
        score_attn = plugin_cache_attn.collect_attn_from_all_blocks(model)[-1, idx_transform, :]    #[B, Q, K]
        score_attn[:,:,idx_transform] = 0 # -> let the current unmask token to be 0

        idx_denoising_current = DenoisingIDXSelector(score_attn)    # 0~64
        idx_denoising_abs = idx_denoising_current + position_start
        idx_current = torch.cat([idx_refresh, idx_denoising_abs])

        logits = model(x[:, idx_current], idx_current=idx_current, shape_target=shape_target).logits
        logits_transform = logits[:, -idx_denoising_abs.shape[-1]:]

        # different here compared to step == 0
        snapshot.update_logits_(idx_denoising_abs, logits_transform)
        conf_snapshot = snapshot.transform_logits(collector)
        # different ends

        if transform_in_h:
            mask_nodenoising_abs = ~torch.isin(idx_block_abs, idx_denoising_abs)
            conf_snapshot.scatter_(0, mask_nodenoising_abs, conf_snapshot)
        # end
    # end

        idx_sorted_by_conf = sorter.argsort(conf_snapshot, snapshot)    # truth
        num_unmask = quota_helper.get_quota(step)
        idx_transform = idx_sorted_by_conf[:, :num_unmask]
        snapshot.materialize_by_idx_(idx_transform, conf_snapshot)
        snapshot.update_this(1, idx_transform, y=x).update_this(1, idx_transform, p_finalized=p_finalized)
    # end
# end










In [ ]:

for id_block in range(num_blocks):
    position_start = len_prompt + id_block * size_block
    position_end = position_start + size_block
    mask_mask_blk = x[:,position_start:position_end] == id_mask
    quota_helper = BlockDiffusionQuotaHelper(mask_mask_blk, size_block)


    score_attn = plugin_cache_attn.collect_attn_from_all_blocks(model)